# 04 - Fine-Tuning with HuggingFace Transformers (OPTIONAL)

---

> **This notebook is OPTIONAL.** It requires the `transformers` library from HuggingFace, which is not a core dependency. All imports are guarded with `try/except` blocks. If the library is not available, expected outputs and pseudocode are shown so you can still follow the concepts.

## Learning Objectives

By the end of this notebook, you will be able to:

- Load a **pretrained Transformer model** (DistilBERT) from HuggingFace
- Use a HuggingFace **tokenizer** for text preprocessing
- **Fine-tune** a pretrained model on a small classification dataset
- Understand the difference between **training from scratch** (Notebook 03) and **fine-tuning**
- Evaluate fine-tuned model performance

## Prerequisites

- Completed Notebooks 01-03 (attention, Transformer architecture, training)
- *Optional:* `pip install transformers datasets` (the notebook works without them)

## Table of Contents

1. [Why Fine-Tuning?](#1-why-fine-tuning)
2. [Setup: Check HuggingFace Availability](#2-setup-check-huggingface-availability)
3. [Loading a Pretrained Model](#3-loading-a-pretrained-model)
4. [Tokenizer Usage](#4-tokenizer-usage)
5. [Preparing the Dataset](#5-preparing-the-dataset)
6. [Fine-Tuning Pipeline](#6-fine-tuning-pipeline)
7. [Code: Complete Fine-Tuning](#7-code-complete-fine-tuning)
8. [Evaluation](#8-evaluation)
9. [Exercise: Fine-Tune on a Different Dataset](#9-exercise-fine-tune-on-a-different-dataset)
10. [Common Mistakes & Debugging Tips](#10-common-mistakes--debugging-tips)

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

set_global_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

---
## 1. Why Fine-Tuning?

**Training from scratch** (Notebook 03):
- Random initialization
- Needs lots of data to learn language patterns
- Vocabulary, embeddings, attention all start from zero

**Fine-tuning a pretrained model:**
- Start with a model already trained on massive text corpora (e.g., Wikipedia, BookCorpus)
- The model already "understands" language (grammar, semantics, world knowledge)
- We only adapt it to our specific task with a small amount of labeled data
- Much faster convergence, often better performance

**Analogy:** Training from scratch is like teaching a child to read *and* classify documents. Fine-tuning is like giving a literate adult a new classification task -- they already know how to read.

---
## 2. Setup: Check HuggingFace Availability

> **OPTIONAL:** This section checks whether the `transformers` library is installed. If not, the notebook will show pseudocode and expected outputs instead.

In [ ]:
# Guard all HuggingFace imports
HF_AVAILABLE = False

try:
    from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
    from transformers import get_linear_schedule_with_warmup
    HF_AVAILABLE = True
    print("HuggingFace Transformers is available!")
    import transformers
    print(f"  transformers version: {transformers.__version__}")
except ImportError:
    print("HuggingFace Transformers is NOT installed.")
    print("To install: pip install transformers")
    print("")
    print("This notebook will show pseudocode and expected outputs.")
    print("You can still follow along to understand the concepts.")

---
## 3. Loading a Pretrained Model

We use **DistilBERT** (`distilbert-base-uncased`) -- a smaller, faster version of BERT that retains 97% of BERT's performance:

- 6 Transformer encoder layers (BERT has 12)
- 66M parameters (BERT-base has 110M)
- Same tokenizer as BERT (WordPiece, 30522 vocabulary)

**OPTIONAL** -- this cell requires `transformers`.

In [ ]:
if HF_AVAILABLE:
    # Load pretrained tokenizer and model
    model_name = 'distilbert-base-uncased'
    tokenizer = DistilBertTokenizer.from_pretrained(model_name)
    
    print(f"Tokenizer vocabulary size: {tokenizer.vocab_size}")
    print(f"Model name: {model_name}")
    
    # Example tokenization
    sample_text = "The Transformer architecture revolutionized NLP."
    tokens = tokenizer.tokenize(sample_text)
    print(f"\nSample text: '{sample_text}'")
    print(f"Tokens: {tokens}")
    print(f"Token IDs: {tokenizer.convert_tokens_to_ids(tokens)}")
else:
    print("--- Pseudocode (HuggingFace not available) ---")
    print()
    print("from transformers import DistilBertTokenizer, DistilBertForSequenceClassification")
    print()
    print("tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')")
    print("# Vocabulary size: 30522")
    print()
    print("# Example tokenization:")
    print("# 'The Transformer architecture revolutionized NLP.'")
    print("# -> ['the', 'transformer', 'architecture', 'revolution', '##ized', 'nl', '##p', '.']")
    print("# -> [1996, 10938, 4294, 4329, 3550, 17953, 2361, 1012]")
    print()
    print("# Note: WordPiece splits unknown words into subwords (## prefix)")

---
## 4. Tokenizer Usage

HuggingFace tokenizers handle the full preprocessing pipeline:

```python
encoded = tokenizer(
    text,
    padding=True,         # pad shorter sequences
    truncation=True,      # truncate longer sequences
    max_length=128,       # max sequence length
    return_tensors="pt"   # return PyTorch tensors
)
```

This returns:
- `input_ids`: Token indices (with special tokens `[CLS]` and `[SEP]`)
- `attention_mask`: 1 for real tokens, 0 for padding

**OPTIONAL** -- this cell requires `transformers`.

In [ ]:
if HF_AVAILABLE:
    # Demonstrate tokenizer usage
    texts = [
        "Space exploration is fascinating.",
        "Baseball is America's pastime."
    ]
    
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    
    print("Encoded keys:", list(encoded.keys()))
    print(f"\ninput_ids shape: {encoded['input_ids'].shape}")
    print(f"attention_mask shape: {encoded['attention_mask'].shape}")
    print()
    
    for i, text in enumerate(texts):
        print(f"Text {i}: '{text}'")
        print(f"  input_ids:      {encoded['input_ids'][i].tolist()}")
        print(f"  attention_mask:  {encoded['attention_mask'][i].tolist()}")
        print(f"  decoded:        {tokenizer.decode(encoded['input_ids'][i])}")
        print()
else:
    print("--- Expected Output (HuggingFace not available) ---")
    print()
    print("Encoded keys: ['input_ids', 'attention_mask']")
    print()
    print("input_ids shape: torch.Size([2, 9])")
    print("attention_mask shape: torch.Size([2, 9])")
    print()
    print("Text 0: 'Space exploration is fascinating.'")
    print("  input_ids:      [101, 2686, 8404, 2003, 12411, 1012, 102, 0, 0]")
    print("  attention_mask:  [1, 1, 1, 1, 1, 1, 1, 0, 0]")
    print("  decoded:        [CLS] space exploration is fascinating. [SEP] [PAD] [PAD]")
    print()
    print("Text 1: 'Baseball is America's pastime.'")
    print("  input_ids:      [101, 3598, 2003, 2637, 1005, 1055, 14946, 1012, 102]")
    print("  attention_mask:  [1, 1, 1, 1, 1, 1, 1, 1, 1]")
    print("  decoded:        [CLS] baseball is america ' s pastime. [SEP]")
    print()
    print("Note: [CLS] (101) and [SEP] (102) are special tokens added by BERT-style models.")
    print("[CLS] token is used for classification tasks.")

---
## 5. Preparing the Dataset

We use the same 4-category 20 Newsgroups subset as Notebook 03, but now preprocess with the HuggingFace tokenizer.

In [ ]:
# Load data (same as Notebook 03)
categories = ['sci.space', 'rec.sport.baseball', 'comp.graphics', 'talk.politics.mideast']

newsgroups = fetch_20newsgroups(
    subset='all',
    categories=categories,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)

all_texts = newsgroups.data
all_labels = newsgroups.target
label_names = newsgroups.target_names

# Use a smaller subset for faster fine-tuning demo
set_global_seed(42)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    all_texts, all_labels, test_size=0.2, random_state=42, stratify=all_labels
)

# Further subsample training data to show fine-tuning works with limited data
TRAIN_SUBSET = 500
train_texts = train_texts[:TRAIN_SUBSET]
train_labels = train_labels[:TRAIN_SUBSET]

print(f"Training samples: {len(train_texts)} (subsampled from full set)")
print(f"Test samples: {len(test_texts)}")
print(f"Classes: {label_names}")

In [ ]:
MAX_LEN = 128  # shorter for speed (DistilBERT supports up to 512)

if HF_AVAILABLE:
    class NewsGroupDataset(Dataset):
        """Dataset for fine-tuning with HuggingFace tokenizer."""
        def __init__(self, texts, labels, tokenizer, max_len=128):
            self.texts = texts
            self.labels = labels
            self.tokenizer = tokenizer
            self.max_len = max_len
        
        def __len__(self):
            return len(self.texts)
        
        def __getitem__(self, idx):
            encoded = self.tokenizer(
                self.texts[idx],
                padding='max_length',
                truncation=True,
                max_length=self.max_len,
                return_tensors='pt'
            )
            return {
                'input_ids': encoded['input_ids'].squeeze(0),
                'attention_mask': encoded['attention_mask'].squeeze(0),
                'labels': torch.tensor(self.labels[idx], dtype=torch.long)
            }
    
    train_dataset = NewsGroupDataset(train_texts, train_labels, tokenizer, MAX_LEN)
    test_dataset = NewsGroupDataset(test_texts, test_labels, tokenizer, MAX_LEN)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
    
    print(f"Train batches: {len(train_loader)}")
    print(f"Test batches: {len(test_loader)}")
    
    sample = train_dataset[0]
    print(f"\nSample input_ids shape: {sample['input_ids'].shape}")
    print(f"Sample attention_mask shape: {sample['attention_mask'].shape}")
else:
    print("--- Dataset Setup (pseudocode) ---")
    print()
    print("class NewsGroupDataset(Dataset):")
    print("    def __init__(self, texts, labels, tokenizer, max_len=128):")
    print("        ...")
    print("    def __getitem__(self, idx):")
    print("        encoded = self.tokenizer(")
    print("            self.texts[idx],")
    print("            padding='max_length',")
    print("            truncation=True,")
    print("            max_length=self.max_len,")
    print("            return_tensors='pt'")
    print("        )")
    print("        return {")
    print("            'input_ids': encoded['input_ids'].squeeze(0),")
    print("            'attention_mask': encoded['attention_mask'].squeeze(0),")
    print("            'labels': torch.tensor(self.labels[idx])")
    print("        }")

---
## 6. Fine-Tuning Pipeline

**Key decisions for fine-tuning:**

- **Learning rate:** Much smaller than training from scratch (2e-5 to 5e-5 is typical)
- **Epochs:** 2-4 is usually sufficient (pretrained weights are already good)
- **Warmup:** Linear warmup for the first 10% of steps prevents early instability
- **Freeze vs. unfreeze:** We fine-tune the entire model (all layers); alternatively, you can freeze early layers

**OPTIONAL** -- The training code below requires `transformers`.

In [ ]:
if HF_AVAILABLE:
    def train_one_epoch(model, dataloader, optimizer, scheduler, device):
        """Train for one epoch."""
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            logits = outputs.logits
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item() * input_ids.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        
        return total_loss / total, correct / total


    def evaluate(model, dataloader, device):
        """Evaluate on a dataset."""
        model.eval()
        total_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
                logits = outputs.logits
                
                total_loss += loss.item() * input_ids.size(0)
                preds = logits.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        
        return total_loss / total, correct / total

    print("Training and evaluation functions defined.")
else:
    print("--- Training Functions (pseudocode) ---")
    print()
    print("# The HuggingFace model returns a NamedTuple with .loss and .logits")
    print("# when labels are provided:")
    print("#   outputs = model(input_ids=..., attention_mask=..., labels=...)")
    print("#   loss = outputs.loss     # CrossEntropy computed internally")
    print("#   logits = outputs.logits # (batch, num_classes)")
    print("#")
    print("# Training loop is standard PyTorch with:")
    print("#   - Very small learning rate (2e-5)")
    print("#   - Linear warmup scheduler")
    print("#   - Gradient clipping (max_norm=1.0)")

---
## 7. Code: Complete Fine-Tuning

**OPTIONAL** -- This cell requires `transformers`.

In [ ]:
if HF_AVAILABLE:
    set_global_seed(42)
    
    NUM_CLASSES = len(label_names)
    NUM_EPOCHS = 3
    LR = 2e-5
    
    # Load pretrained model with classification head
    model = DistilBertForSequenceClassification.from_pretrained(
        'distilbert-base-uncased',
        num_labels=NUM_CLASSES
    ).to(device)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print()
    
    # Optimizer and scheduler
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * NUM_EPOCHS
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )
    
    print(f"Total training steps: {total_steps}")
    print(f"Warmup steps: {warmup_steps}")
    print(f"Training for {NUM_EPOCHS} epochs...")
    print()
    
    history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}
    
    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, scheduler, device
        )
        test_loss, test_acc = evaluate(model, test_loader, device)
        
        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}")
    
    print(f"\nFinal Test Accuracy: {history['test_acc'][-1]:.4f}")

else:
    print("--- Expected Output (HuggingFace not available) ---")
    print()
    print("Total parameters: 66,955,780")
    print("Trainable parameters: 66,955,780")
    print()
    print("Total training steps: 96")
    print("Warmup steps: 9")
    print("Training for 3 epochs...")
    print()
    print("Epoch 1/3 | Train Loss: 0.9832 Acc: 0.6240 | Test Loss: 0.5421 Acc: 0.8134")
    print("Epoch 2/3 | Train Loss: 0.3715 Acc: 0.8900 | Test Loss: 0.4103 Acc: 0.8657")
    print("Epoch 3/3 | Train Loss: 0.1802 Acc: 0.9540 | Test Loss: 0.3891 Acc: 0.8790")
    print()
    print("Final Test Accuracy: 0.8790")
    print()
    print("Note: With only 500 training samples, DistilBERT fine-tuning achieves ~88% accuracy,")
    print("which is significantly better than training a small Transformer from scratch.")
    print("This demonstrates the power of transfer learning with pretrained models.")

---
## 8. Evaluation

**OPTIONAL** -- this cell requires `transformers`.

In [ ]:
if HF_AVAILABLE:
    # Get predictions
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=1).cpu()
            all_preds.append(preds)
            all_labels.append(batch['labels'])
    
    preds = torch.cat(all_preds).numpy()
    true = torch.cat(all_labels).numpy()
    
    print("Classification Report (Fine-Tuned DistilBERT):")
    print(classification_report(true, preds, target_names=label_names))
    
    # Plot training curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs_range = range(1, NUM_EPOCHS + 1)
    
    axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train')
    axes[0].plot(epochs_range, history['test_loss'], 'r-o', label='Test')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(epochs_range, history['train_acc'], 'b-o', label='Train')
    axes[1].plot(epochs_range, history['test_acc'], 'r-o', label='Test')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle('DistilBERT Fine-Tuning Curves', fontsize=14)
    plt.tight_layout()
    plt.show()

else:
    print("--- Expected Classification Report ---")
    print()
    print("                          precision    recall  f1-score   support")
    print("")
    print("          comp.graphics       0.87      0.85      0.86       195")
    print("    rec.sport.baseball        0.90      0.91      0.90       199")
    print("             sci.space        0.86      0.89      0.87       198")
    print(" talk.politics.mideast        0.88      0.85      0.86       183")
    print("")
    print("                accuracy                          0.88       775")
    print("               macro avg      0.88      0.87      0.87       775")
    print("            weighted avg      0.88      0.88      0.88       775")
    print()
    print("Key takeaway: Fine-tuning achieves strong results with only 500 training samples,")
    print("because the pretrained model already understands English language structure.")

---
## 9. Exercise: Fine-Tune on a Different Dataset

**Task:** Fine-tune DistilBERT on a different subset of 20 Newsgroups categories.

1. Choose 3-5 categories from: `'alt.atheism'`, `'sci.med'`, `'soc.religion.christian'`, `'comp.os.ms-windows.misc'`, `'rec.autos'`
2. Load the data and create train/test splits
3. Fine-tune DistilBERT and evaluate
4. Compare accuracy with the 4-category task above

```python
# Your code here (requires transformers library)
#
# new_categories = ['alt.atheism', 'sci.med', 'soc.religion.christian']
# new_data = fetch_20newsgroups(subset='all', categories=new_categories, ...)
# ...
# Fine-tune and evaluate using the same pipeline above
```

**If `transformers` is not installed**, answer these conceptual questions:

1. Why might categories like `alt.atheism` and `soc.religion.christian` be harder to separate than our 4 categories?
2. Would you expect fine-tuning to help more or less on harder classification tasks?
3. What would happen if you froze the pretrained layers and only trained the classification head?

In [ ]:
# Try the exercise yourself before looking at the solution!





### Solution (Conceptual)

1. **Why harder?** `alt.atheism` and `soc.religion.christian` share vocabulary (religion, god, belief). The model must learn subtle semantic differences, not just keyword matching.

2. **Fine-tuning helps more on harder tasks.** Pretrained models capture nuanced semantics that are critical when categories overlap. A model trained from scratch on limited data cannot learn these distinctions.

3. **Frozen pretrained layers (linear probing):** Only the classification head learns. This is faster but less accurate, because the pretrained representations may not perfectly align with the new task. Fine-tuning the full model allows it to adjust internal representations for the specific task.

---
## 10. Common Mistakes & Debugging Tips

**1. Learning rate too high for fine-tuning**
- Pretrained weights are carefully tuned; a large LR (e.g., 1e-3) will destroy them
- Use 2e-5 to 5e-5 for full fine-tuning
- Symptom: loss spikes in the first few steps, then performance is worse than random

**2. Forgetting `attention_mask`**
- Without the attention mask, padding tokens attend to (and are attended by) real tokens
- Always pass `attention_mask` to the model

**3. Tokenizer mismatch**
- Always use the tokenizer that matches the model (e.g., `DistilBertTokenizer` for `distilbert-base-uncased`)
- Using the wrong tokenizer produces nonsensical token IDs

**4. Not using `model.eval()` during evaluation**
- Dropout and other regularization layers behave differently in train vs eval mode
- Always call `model.eval()` before evaluation and `model.train()` before training

**5. Running out of GPU memory**
- DistilBERT has 66M parameters; long sequences on small GPUs can cause OOM
- Solutions: reduce `max_length`, reduce batch size, use gradient accumulation
- `torch.cuda.empty_cache()` can help free unused memory

**6. Overfitting with few training samples**
- With very small datasets (< 100 samples), fine-tuning all layers can overfit
- Consider: freeze early layers, use smaller learning rate, or use fewer epochs

**7. `cased` vs `uncased` model mismatch**
- `uncased` models expect lowercased input (tokenizer handles this)
- Mixing `cased` tokenizer with `uncased` model (or vice versa) leads to poor results

---

**Next notebook:** [05 - Embeddings and Semantic Search (Optional)](05_Embeddings_and_Semantic_Search_optional.ipynb) -- explore word and sentence embeddings for semantic similarity and search.